# 06 · Grounded conversation, semantic retrieval and speech

TA Units 2–4. Speech is transcribed to text; retrieval selects real catalog records; structured filters enforce constraints; an optional local transformer phrases the response. The default app is catalog retrieval, not a pretend trained LLM.

In [1]:
from pathlib import Path
import sys, json, os
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Open this notebook from CineScope/notebooks or CineScope.'
sys.path.insert(0, str(ROOT / 'backend'))
sys.path.insert(0, str(ROOT / 'scripts'))
os.chdir(ROOT)
(ROOT / 'artifacts').mkdir(exist_ok=True)
SEED = 42

In [2]:
from cinescope.chat import MovieChat
from cinescope.recommender import Recommender
from cinescope.reviews import ReviewAnalyzer
from cinescope.storage import Store
import tempfile
movies=json.loads((ROOT/'data/demo/catalog.json').read_text())
conversation_dir=tempfile.TemporaryDirectory()
assistant=MovieChat(Recommender(movies),Store(conversation_dir.name),ReviewAnalyzer(mode='lexicon'))
context={}
for message in ['Sci-fi with romance','Something under 120 minutes','Summarize the reviews for Arrival','What about its pacing?']:
    response=assistant.respond(message,context=context)
    context=response['context']
    print('YOU:',message,'\nCINE:',response['answer'],'\n')

YOU: Sci-fi with romance 
CINE: Here are 3 films matching Sci-Fi + Romance. Eternal Sunshine of the Spotless Mind: memory, love. The Last Signal: space, love. Her: artificial intelligence, loneliness. 

YOU: Something under 120 minutes 
CINE: Here are 2 films matching Sci-Fi + Romance. Eternal Sunshine of the Spotless Mind: memory, love. The Last Signal: space, love. 

YOU: Summarize the reviews for Arrival 
CINE: Arrival: Across 3 eligible reviews, 2 lean positive and 1 lean negative; 0 are uncertain. Praise centers on visuals, music, acting. This is a small sample. These are synthetic sample reviews. 

YOU: What about its pacing? 
CINE: Arrival: Across 3 eligible reviews, 2 lean positive and 1 lean negative; 0 are uncertain. Praise centers on visuals, music, acting. This is a small sample. These are synthetic sample reviews. 



## Add actual transformers
Download models explicitly on a machine with access to the model hub. Keep the model revisions in your experiment record. The server only loads local paths; no download occurs in the middle of a user request.

`all-MiniLM-L6-v2` embeds the query and catalog. `flan-t5-base` optionally rewrites retrieved facts. Its output remains experimental and should be inspected for unsupported claims. Authoritative title cards and extractive review summaries remain separate from generated wording.

In [3]:
DOWNLOAD_MODELS = False
if DOWNLOAD_MODELS:
    from sentence_transformers import SentenceTransformer
    from transformers import AutoTokenizer,AutoModelForSeq2SeqLM
    encoder=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    encoder.save(str(ROOT/'models/movie_embeddings'))
    name='google/flan-t5-base'
    AutoTokenizer.from_pretrained(name).save_pretrained(ROOT/'models/movie_chat')
    AutoModelForSeq2SeqLM.from_pretrained(name).save_pretrained(ROOT/'models/movie_chat')
    print('Set CINESCOPE_EMBEDDING_MODEL and CINESCOPE_CHAT_MODEL to those local directories, then restart the API.')
else:
    print('No transformer weights downloaded in demo mode.')

No transformer weights downloaded in demo mode.


In [4]:
# Optional server-side speech path. The frontend also supports browser speech recognition.
DOWNLOAD_WHISPER = False
AUDIO_FILE = ROOT/'data/my_prompt.wav'
if DOWNLOAD_WHISPER:
    from huggingface_hub import snapshot_download
    path=snapshot_download('Systran/faster-whisper-small.en',local_dir=str(ROOT/'models/whisper-small.en'))
    print('Set CINESCOPE_WHISPER_MODEL='+path)
if AUDIO_FILE.exists() and (ROOT/'models/whisper-small.en').exists():
    from faster_whisper import WhisperModel
    whisper=WhisperModel(str(ROOT/'models/whisper-small.en'),device='cpu',compute_type='int8',local_files_only=True)
    segments,info=whisper.transcribe(str(AUDIO_FILE))
    transcript=' '.join(s.text for s in segments)
    print(transcript)
    print(assistant.respond(transcript)['answer'])
else:
    print('Speech not run: supply a local audio file and downloaded Whisper model.')

Speech not run: supply a local audio file and downloaded Whisper model.


## Evaluation
Prepare natural-language queries with human-labeled relevant movies and explicit constraints (genres, exclusions, time limit). Measure retrieval recall/NDCG, constraint satisfaction and response source fidelity. For STT, compare the transcript with a human reference using word error rate. Browser speech availability depends on browser support, microphone permissions and potentially the browser vendor's online service; typing is always available.

The `context` JSON carries conversational constraints and the last referenced movie across turns. Start a new chat or say “reset” to clear it. Review summaries use the linked movie ID and evidence IDs, not the LLM's memory.